In [1]:
from create_inverse_dynamics_dataset import INVERSE_DYNAMICS_SYSTEM_PROMPT
from typing import Dict, List
import json

In [2]:


def trajectory_to_inverse_dynamics_samples(traj: Dict) -> List[Dict]:
    """
    Convert trajectory to inverse dynamics samples.

    Format: [mem_t] + [mem_{t+1}] → action_t

    For each pair of consecutive states (t, t+1), predict action_t.
    A trajectory with n rounds produces n-1 samples.
    """
    samples = []
    rounds = traj['rounds']
    task_inst = traj['task_instruction']
    n = len(rounds)

    if n < 2:
        return samples

    # Iterate over all consecutive state pairs
    for t in range(n - 1):
        mem_t = rounds[t]['observation']
        mem_t_plus_1 = rounds[t + 1]['observation']
        action_t = rounds[t]['action']

        messages = [
            {'role': 'system', 'content': INVERSE_DYNAMICS_SYSTEM_PROMPT},
            {'role': 'user', 'content': [
                {'type': 'text', 'text': f'Task Instruction: {task_inst}\n\n'},
                {'type': 'text', 'text': f'Round {t} observation:'},
                {'type': 'memory_text', 'memory_text': {'text': mem_t}},
                {'type': 'text', 'text': f'\n\nRound {t + 1} observation:'},
                {'type': 'memory_text', 'memory_text': {'text': mem_t_plus_1}},
                {'type': 'text', 'text': f'\n\nWhat action was taken after Round {t}?'},
            ]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': action_t}]},
        ]

        samples.append({
            'messages': messages,
            'type': 'inverse_dynamics',
            'round_t': t,
            'round_t_plus_1': t + 1,
            'total_rounds': n,
        })

    return samples

In [3]:
# 加载原始trajectories并分析分布
import json
from collections import Counter

with open('webarena_merged_trajectories.json', 'r') as f:
    trajectories = json.load(f)

print(f"Total trajectories: {len(trajectories)}")

# 分析trajectory长度分布
traj_lengths = [len(t['rounds']) for t in trajectories]
print(f"\nTrajectory length distribution:")
print(f"  Min: {min(traj_lengths)}, Max: {max(traj_lengths)}, Mean: {sum(traj_lengths)/len(traj_lengths):.1f}")
print(f"  Length counts: {dict(sorted(Counter(traj_lengths).items()))}")

Total trajectories: 1114

Trajectory length distribution:
  Min: 0, Max: 52, Mean: 8.1
  Length counts: {0: 1, 1: 41, 2: 6, 3: 59, 4: 59, 5: 76, 6: 124, 7: 122, 8: 155, 9: 202, 10: 104, 11: 67, 12: 24, 13: 12, 14: 16, 15: 3, 16: 4, 17: 5, 18: 1, 19: 2, 20: 3, 21: 2, 22: 2, 23: 1, 24: 1, 25: 1, 26: 3, 27: 13, 28: 2, 33: 1, 44: 1, 52: 1}


In [ ]:
# 分析inverse dynamics样本中round_t的分布
all_inv_samples = []
for traj in trajectories:
    all_inv_samples.extend(trajectory_to_inverse_dynamics_samples(traj))

print(f"Total inverse dynamics samples: {len(all_inv_samples)}")

# Round t 分布
round_t_dist = Counter(s['round_t'] for s in all_inv_samples)
print(f"\nRound t distribution (before resampling):")
for k, v in sorted(round_t_dist.items()):
    print(f"  round_t={k}: {v} samples ({v/len(all_inv_samples)*100:.1f}%)")

# Action类型分布 - 修正正则表达式
import re
def extract_action_type(action_str):
    # 格式: do(action="Click", ...) 或 do(action="Scroll Down")
    match = re.search(r'do\(action=["\']([^"\']+)["\']', action_str)
    if match:
        return match.group(1)
    if 'exit(' in action_str:
        return 'exit'
    if 'go_backward' in action_str:
        return 'go_backward'
    if 'go_forward' in action_str:
        return 'go_forward'
    return 'unknown'

action_types = [extract_action_type(s['messages'][2]['content'][0]['text']) for s in all_inv_samples]
action_dist = Counter(action_types)
print(f"\nAction type distribution:")
for k, v in sorted(action_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v} ({v/len(all_inv_samples)*100:.1f}%)")

In [ ]:
# 清洗action为自然语言
import re

def clean_action_to_natural_language(action_str: str) -> str:
    """
    将action字符串转换为自然语言描述。
    - 去掉 element="数字" 
    - 保留注释中的语义描述
    - quote() 转为自然语言
    """
    
    # 处理 quote() - 转为自然语言
    quote_match = re.search(r'quote\(content=["\'](.+?)["\']\)', action_str, re.DOTALL)
    if quote_match:
        content = quote_match.group(1)
        return f"Record the following information: {content}"
    
    # 处理 go_backward / go_forward
    if 'go_backward()' in action_str:
        return "Go back to the previous page."
    if 'go_forward()' in action_str:
        return "Go forward to the next page."
    
    # 提取注释中的元素描述 (# Element: ...)
    element_desc = ""
    comment_match = re.search(r'#\s*Element:\s*(.+?)(?:\n|$)', action_str)
    if comment_match:
        element_desc = comment_match.group(1).strip()
    
    # 解析 do() 调用
    action_match = re.search(r'do\(action=["\']([^"\']+)["\']', action_str)
    if not action_match:
        return action_str  # 无法解析，返回原始
    
    action_type = action_match.group(1)
    
    # 提取 argument (如果有)
    arg_match = re.search(r'argument=["\']([^"\']*)["\']', action_str)
    argument = arg_match.group(1) if arg_match else None
    
    # 根据action类型生成自然语言
    if action_type == "Click":
        if element_desc:
            return f"Click on {element_desc}."
        return "Click on the element."
    
    elif action_type == "Type":
        if element_desc and argument:
            return f"Type \"{argument}\" into {element_desc}."
        elif argument:
            return f"Type \"{argument}\"."
        return "Type text into the field."
    
    elif action_type == "Search":
        if element_desc and argument:
            return f"Search for \"{argument}\" in {element_desc}."
        elif argument:
            return f"Search for \"{argument}\"."
        return "Perform a search."
    
    elif action_type == "Hover":
        if element_desc:
            return f"Hover over {element_desc}."
        return "Hover over the element."
    
    elif action_type == "Select Dropdown Option":
        if element_desc and argument:
            return f"Select \"{argument}\" from {element_desc}."
        elif argument:
            return f"Select dropdown option \"{argument}\"."
        return "Select a dropdown option."
    
    elif action_type == "Scroll Down":
        return "Scroll down the page."
    
    elif action_type == "Scroll Up":
        return "Scroll up the page."
    
    elif action_type == "Press Enter":
        return "Press the Enter key."
    
    elif action_type == "Wait":
        return "Wait for the page to load."
    
    else:
        # 未知action类型
        if element_desc:
            return f"{action_type} on {element_desc}."
        return f"{action_type}."

# 应用清洗
cleaned_samples = []
for s in all_inv_samples:
    new_s = s.copy()
    new_s['messages'] = json.loads(json.dumps(s['messages']))  # deep copy
    
    # 清洗assistant回复中的action
    original_action = new_s['messages'][2]['content'][0]['text']
    cleaned_action = clean_action_to_natural_language(original_action)
    new_s['messages'][2]['content'][0]['text'] = cleaned_action
    new_s['action_type'] = extract_action_type(original_action)
    
    cleaned_samples.append(new_s)

print(f"Cleaned {len(cleaned_samples)} samples")

# 查看清洗后的分布
cleaned_action_types = [s['action_type'] for s in cleaned_samples]
cleaned_dist = Counter(cleaned_action_types)
print(f"\nAction type distribution after cleaning:")
for k, v in sorted(cleaned_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v} ({v/len(cleaned_samples)*100:.1f}%)")

In [6]:
# 查看实际的action格式
print("Sample action strings:")
for i, s in enumerate(all_inv_samples[:10]):
    action_str = s['messages'][2]['content'][0]['text']
    print(f"{i}: {action_str[:200]}")
    print()

Sample action strings:
0: # Element: the 'Forums' link at the top center
do(action="Click", element="1")

1: # Element: the 'Create forum' button next to the 'List of forums' title
do(action="Click", element="17")

2: # Element: the 'Name' field at the top of the page
do(action="Type", argument="VirtualRealityVanguard", element="12")

3: # Element: the 'Title' input field in the middle of the page
do(action="Type", argument="VirtualRealityVanguard", element="14")

4: # Element: the description text area under the 'Title' field
do(action="Type", argument="Cutting-edge forum for VR enthusiasts to discuss the latest trends, games, and applications in the virtual real

5: # Element: the 'Sidebar' field at the bottom of the page
do(action="Type", argument="virtualreality, technology, trends, gaming", element="18")

6: do(action="Scroll Down")

7: # Element: the 'Create forum' button
do(action="Click", element="10")

8: # Element: the 'REPORTS' section on the left sidebar
do(action="Click", 

In [7]:
# 重新运行分析
from collections import Counter
import re

# Round t 分布
round_t_dist = Counter(s['round_t'] for s in all_inv_samples)
print(f"Total inverse dynamics samples: {len(all_inv_samples)}")
print(f"\nRound t distribution:")
for k, v in sorted(round_t_dist.items()):
    print(f"  round_t={k}: {v} ({v/len(all_inv_samples)*100:.1f}%)")

# Action类型分布 - 修正正则表达式
def extract_action_type(action_str):
    # 格式: do(action="Click", ...) 或 do(action="Scroll Down")
    match = re.search(r'do\(action=["\']([^"\']+)["\']', action_str)
    if match:
        return match.group(1)
    if 'exit(' in action_str:
        return 'exit'
    if 'go_backward' in action_str:
        return 'go_backward'
    if 'go_forward' in action_str:
        return 'go_forward'
    return 'unknown'

action_types = [extract_action_type(s['messages'][2]['content'][0]['text']) for s in all_inv_samples]
action_dist = Counter(action_types)
print(f"\nAction type distribution:")
for k, v in sorted(action_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v} ({v/len(all_inv_samples)*100:.1f}%)")

Total inverse dynamics samples: 7916

Round t distribution:
  round_t=0: 1072 (13.5%)
  round_t=1: 1066 (13.5%)
  round_t=2: 1007 (12.7%)
  round_t=3: 948 (12.0%)
  round_t=4: 872 (11.0%)
  round_t=5: 748 (9.4%)
  round_t=6: 626 (7.9%)
  round_t=7: 471 (5.9%)
  round_t=8: 269 (3.4%)
  round_t=9: 165 (2.1%)
  round_t=10: 98 (1.2%)
  round_t=11: 74 (0.9%)
  round_t=12: 62 (0.8%)
  round_t=13: 46 (0.6%)
  round_t=14: 43 (0.5%)
  round_t=15: 39 (0.5%)
  round_t=16: 34 (0.4%)
  round_t=17: 33 (0.4%)
  round_t=18: 31 (0.4%)
  round_t=19: 28 (0.4%)
  round_t=20: 26 (0.3%)
  round_t=21: 24 (0.3%)
  round_t=22: 23 (0.3%)
  round_t=23: 22 (0.3%)
  round_t=24: 21 (0.3%)
  round_t=25: 18 (0.2%)
  round_t=26: 5 (0.1%)
  round_t=27: 3 (0.0%)
  round_t=28: 3 (0.0%)
  round_t=29: 3 (0.0%)
  round_t=30: 3 (0.0%)
  round_t=31: 3 (0.0%)
  round_t=32: 2 (0.0%)
  round_t=33: 2 (0.0%)
  round_t=34: 2 (0.0%)
  round_t=35: 2 (0.0%)
  round_t=36: 2 (0.0%)
  round_t=37: 2 (0.0%)
  round_t=38: 2 (0.0%)
  round_t

In [8]:
# 查看unknown action的具体内容
unknown_samples = [s for s in all_inv_samples if extract_action_type(s['messages'][2]['content'][0]['text']) == 'unknown']
print(f"Unknown samples: {len(unknown_samples)}\n")

for i, s in enumerate(unknown_samples[:20]):
    action_str = s['messages'][2]['content'][0]['text']
    print(f"{i}: {action_str[:300]}")
    print()

Unknown samples: 45

0: quote(content="The price of order number 000000167, which is dated 7/8/22, is $40.16. The current total price is $40.16.")

1: quote(content="The price of order number 000000181, which is dated 5/22/22, is $298.65. The price of order number 000000168, which is dated 4/27/22, is $24.86. The price of order number 000000159, which is dated 4/5/22, is $77.96. The price of order number 000000169, which is dated 3/10/22, is $206.

2: quote(content="The price of order number 000000154 is $97.15. The price of order number 000000184 is $20.49. The price of order number 000000162 is $53.29. The price of order number 000000174 is $32.47. The price of order number 000000164 is $218.17. The price of order number 000000171 is $133.07. T

3: quote(content="The price of order number 000000176 is $845.07. The current total price is $606.58 + $845.07 = $1451.65.")

4: quote(content="The price of order number 000000178 is $345.84. The price of order number 000000185 is $18.99. The

In [9]:
# 查看各种action的完整格式
from collections import defaultdict

action_examples = defaultdict(list)
for s in all_inv_samples:
    action_str = s['messages'][2]['content'][0]['text']
    action_type = extract_action_type(action_str)
    if len(action_examples[action_type]) < 3:
        action_examples[action_type].append(action_str)

print("Action format examples:\n")
for action_type, examples in sorted(action_examples.items()):
    print(f"=== {action_type} ===")
    for ex in examples:
        print(f"  {ex[:200]}")
    print()

Action format examples:

=== Click ===
  # Element: the 'Forums' link at the top center
do(action="Click", element="1")
  # Element: the 'Create forum' button next to the 'List of forums' title
do(action="Click", element="17")
  # Element: the 'Create forum' button
do(action="Click", element="10")

=== Hover ===
  # Element: the 'Home & Kitchen' category link
do(action="Hover", element="14")
  # Element: the 'Home Décor Products' category link
do(action="Hover", element="22")
  # Element: the 'Grocery & Gourmet Food' category link
do(action="Hover", element="31")

=== Press Enter ===
  do(action="Press Enter")
  do(action="Press Enter")

=== Scroll Down ===
  do(action="Scroll Down")
  do(action="Scroll Down")
  do(action="Scroll Down")

=== Scroll Up ===
  do(action="Scroll Up")
  do(action="Scroll Up")
  do(action="Scroll Up")

=== Search ===
  # Element: the search bar on the top left
do(action="Search", argument="timeit", element="5")
  # Element: the input field labeled 'Select du

In [10]:
import re

def clean_action_to_natural_language(action_str: str) -> str:
    """
    将action字符串转换为自然语言描述。
    - 去掉 element="数字" 
    - 保留注释中的语义描述
    - quote() 转为自然语言
    """
    
    # 处理 quote() - 转为自然语言
    quote_match = re.search(r'quote\(content=["\'](.+?)["\']\)', action_str, re.DOTALL)
    if quote_match:
        content = quote_match.group(1)
        return f"Record the following information: {content}"
    
    # 处理 go_backward / go_forward
    if 'go_backward()' in action_str:
        return "Go back to the previous page."
    if 'go_forward()' in action_str:
        return "Go forward to the next page."
    
    # 提取注释中的元素描述 (# Element: ...)
    element_desc = ""
    comment_match = re.search(r'#\s*Element:\s*(.+?)(?:\n|$)', action_str)
    if comment_match:
        element_desc = comment_match.group(1).strip()
    
    # 解析 do() 调用
    action_match = re.search(r'do\(action=["\']([^"\']+)["\']', action_str)
    if not action_match:
        return action_str  # 无法解析，返回原始
    
    action_type = action_match.group(1)
    
    # 提取 argument (如果有)
    arg_match = re.search(r'argument=["\']([^"\']*)["\']', action_str)
    argument = arg_match.group(1) if arg_match else None
    
    # 根据action类型生成自然语言
    if action_type == "Click":
        if element_desc:
            return f"Click on {element_desc}."
        return "Click on the element."
    
    elif action_type == "Type":
        if element_desc and argument:
            return f"Type \"{argument}\" into {element_desc}."
        elif argument:
            return f"Type \"{argument}\"."
        return "Type text into the field."
    
    elif action_type == "Search":
        if element_desc and argument:
            return f"Search for \"{argument}\" in {element_desc}."
        elif argument:
            return f"Search for \"{argument}\"."
        return "Perform a search."
    
    elif action_type == "Hover":
        if element_desc:
            return f"Hover over {element_desc}."
        return "Hover over the element."
    
    elif action_type == "Select Dropdown Option":
        if element_desc and argument:
            return f"Select \"{argument}\" from {element_desc}."
        elif argument:
            return f"Select dropdown option \"{argument}\"."
        return "Select a dropdown option."
    
    elif action_type == "Scroll Down":
        return "Scroll down the page."
    
    elif action_type == "Scroll Up":
        return "Scroll up the page."
    
    elif action_type == "Press Enter":
        return "Press the Enter key."
    
    elif action_type == "Wait":
        return "Wait for the page to load."
    
    else:
        # 未知action类型
        if element_desc:
            return f"{action_type} on {element_desc}."
        return f"{action_type}."

# 测试
print("=== Testing clean_action_to_natural_language ===\n")
test_cases = [
    '# Element: the \'Forums\' link at the top center\ndo(action="Click", element="1")',
    '# Element: the \'Name\' field at the top of the page\ndo(action="Type", argument="VirtualRealityVanguard", element="12")',
    'do(action="Scroll Down")',
    '# Element: the search bar\ndo(action="Search", argument="timeit", element="5")',
    '# Element: the \'Period\' dropdown\ndo(action="Select Dropdown Option", argument="Month", element="25")',
    'go_backward()',
    'quote(content="The price of order number 000000167 is $40.16.")',
    '# Element: the \'Home & Kitchen\' category link\ndo(action="Hover", element="14")',
]

for tc in test_cases:
    print(f"Original: {tc[:80]}...")
    print(f"Cleaned:  {clean_action_to_natural_language(tc)}")
    print()

=== Testing clean_action_to_natural_language ===

Original: # Element: the 'Forums' link at the top center
do(action="Click", element="1")...
Cleaned:  Click on the 'Forums' link at the top center.

Original: # Element: the 'Name' field at the top of the page
do(action="Type", argument="V...
Cleaned:  Type "VirtualRealityVanguard" into the 'Name' field at the top of the page.

Original: do(action="Scroll Down")...
Cleaned:  Scroll down the page.

Original: # Element: the search bar
do(action="Search", argument="timeit", element="5")...
Cleaned:  Search for "timeit" in the search bar.

Original: # Element: the 'Period' dropdown
do(action="Select Dropdown Option", argument="M...
Cleaned:  Select "Month" from the 'Period' dropdown.

Original: go_backward()...
Cleaned:  Go back to the previous page.

Original: quote(content="The price of order number 000000167 is $40.16.")...
Cleaned:  Record the following information: The price of order number 000000167 is $40.16.

Original: # Element: th

In [11]:
# 应用清洗
cleaned_samples = []
for s in all_inv_samples:
    new_s = s.copy()
    new_s['messages'] = json.loads(json.dumps(s['messages']))  # deep copy
    
    # 清洗assistant回复中的action
    original_action = new_s['messages'][2]['content'][0]['text']
    cleaned_action = clean_action_to_natural_language(original_action)
    new_s['messages'][2]['content'][0]['text'] = cleaned_action
    new_s['action_type'] = extract_action_type(original_action)
    
    cleaned_samples.append(new_s)

print(f"Cleaned {len(cleaned_samples)} samples")

# 查看清洗后的分布
cleaned_action_types = [s['action_type'] for s in cleaned_samples]
cleaned_dist = Counter(cleaned_action_types)
print(f"\nAction type distribution after cleaning:")
for k, v in sorted(cleaned_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v} ({v/len(cleaned_samples)*100:.1f}%)")

# 查看几个清洗后的样本
print("\n=== Sample cleaned actions ===")
for i in [0, 100, 500, 1000, 2000]:
    print(f"[{i}] {cleaned_samples[i]['messages'][2]['content'][0]['text']}")

Cleaned 7916 samples

Action type distribution after cleaning:
  Click: 4716 (59.6%)
  Type: 1384 (17.5%)
  Scroll Down: 896 (11.3%)
  Select Dropdown Option: 360 (4.5%)
  Search: 303 (3.8%)
  go_backward: 120 (1.5%)
  Hover: 72 (0.9%)
  unknown: 45 (0.6%)
  Scroll Up: 17 (0.2%)
  Press Enter: 2 (0.0%)
  Wait: 1 (0.0%)

=== Sample cleaned actions ===
[0] Click on the 'Forums' link at the top center.
[100] Type "Chaz Kangeroo Hoodie" into the 'Name' input field.
[500] Type "Bennington" into The field at the top left is the 'To point' field..
[1000] Go back to the previous page.
[2000] Scroll down the page.


In [12]:
# 查看Click类型的具体内容分布
click_samples = [s for s in cleaned_samples if s['action_type'] == 'Click']
click_actions = [s['messages'][2]['content'][0]['text'] for s in click_samples]

click_action_dist = Counter(click_actions)
print(f"Total Click samples: {len(click_samples)}")
print(f"Unique Click actions: {len(click_action_dist)}")

print(f"\nTop 30 most common Click actions:")
for action, count in click_action_dist.most_common(30):
    print(f"  {count:4d}x: {action[:80]}")

Total Click samples: 4716
Unique Click actions: 1315

Top 30 most common Click actions:
   629x: Click on The Go button.
   187x: Click on the direction sign on the right side of the Go icon.
   116x: Click on the 'Forums' link at the top center.
   110x: Click on the 'Hot' button in the navigation menu.
   108x: Click on the 'Alphabetical' navigation link.
    95x: Click on the 'Alphabetical' navigation link at the top center.
    82x: Click on the 'My Orders' link.
    64x: Click on the 'Forums' link.
    59x: Click on the 'Next page' link.
    46x: Click on the 'Filters' button.
    45x: Click on the 'Next Page' link.
    44x: Click on the 'Apply Filters' button.
    44x: Click on the 'Apply Filters' button next to the 'Cancel' button.
    39x: Click on the 'All time' filter option.
    34x: Click on the 'Create submission' button.
    30x: Click on the 'Search' button.
    29x: Click on the 'Hot' button in the navigation menu at the top center.
    27x: Click on the 'My Account' li

In [13]:
# 分析重复程度
print("Click action repetition analysis:")
print(f"  Total: {len(click_samples)}")
print(f"  Unique: {len(click_action_dist)}")
print(f"  Repetition ratio: {len(click_samples)/len(click_action_dist):.1f}x average")

# 按重复次数分桶
repeat_buckets = {
    '1次 (unique)': 0,
    '2-5次': 0,
    '6-20次': 0,
    '21-50次': 0,
    '51-100次': 0,
    '>100次': 0,
}

for action, count in click_action_dist.items():
    if count == 1:
        repeat_buckets['1次 (unique)'] += count
    elif count <= 5:
        repeat_buckets['2-5次'] += count
    elif count <= 20:
        repeat_buckets['6-20次'] += count
    elif count <= 50:
        repeat_buckets['21-50次'] += count
    elif count <= 100:
        repeat_buckets['51-100次'] += count
    else:
        repeat_buckets['>100次'] += count

print(f"\nSample count by repetition level:")
for bucket, count in repeat_buckets.items():
    print(f"  {bucket}: {count} samples ({count/len(click_samples)*100:.1f}%)")

# 看看其他action类型的重复情况
print("\n=== Other action types repetition ===")
for action_type in ['Type', 'Scroll Down', 'Search', 'Select Dropdown Option']:
    type_samples = [s for s in cleaned_samples if s['action_type'] == action_type]
    type_actions = [s['messages'][2]['content'][0]['text'] for s in type_samples]
    type_dist = Counter(type_actions)
    print(f"{action_type}: {len(type_samples)} total, {len(type_dist)} unique, top={type_dist.most_common(1)[0] if type_dist else 'N/A'}")

Click action repetition analysis:
  Total: 4716
  Unique: 1315
  Repetition ratio: 3.6x average

Sample count by repetition level:
  1次 (unique): 916 samples (19.4%)
  2-5次: 805 samples (17.1%)
  6-20次: 962 samples (20.4%)
  21-50次: 583 samples (12.4%)
  51-100次: 300 samples (6.4%)
  >100次: 1150 samples (24.4%)

=== Other action types repetition ===
Type: 1384 total, 1285 unique, top=('Type "hello1234" into the password input field.', 27)
Scroll Down: 896 total, 1 unique, top=('Scroll down the page.', 896)
Search: 303 total, 263 unique, top=('Search for "a11yproject.com" in the search bar on the top left.', 9)
Select Dropdown Option: 360 total, 90 unique, top=('Select "Foot" from The menu that drops down.', 29)


In [14]:
# 检查 "Click on The Go button." 的obs是否不同
go_button_samples = [s for s in cleaned_samples if s['messages'][2]['content'][0]['text'] == 'Click on The Go button.']

print(f"Total 'Click on The Go button.' samples: {len(go_button_samples)}")

# 提取每个样本的 mem_t 和 mem_t+1
obs_pairs = []
for s in go_button_samples:
    user_content = s['messages'][1]['content']
    # mem_t 在 index 2, mem_t+1 在 index 4
    mem_t = user_content[2]['memory_text']['text']
    mem_t_plus_1 = user_content[4]['memory_text']['text']
    obs_pairs.append((mem_t[:200], mem_t_plus_1[:200]))  # 截取前200字符比较

# 检查unique obs pairs
unique_pairs = set(obs_pairs)
print(f"Unique (mem_t, mem_t+1) pairs: {len(unique_pairs)}")

# 查看几个例子
print("\n=== Sample obs pairs for 'Click on The Go button.' ===")
for i in [0, 100, 300, 500]:
    if i < len(go_button_samples):
        s = go_button_samples[i]
        user_content = s['messages'][1]['content']
        mem_t = user_content[2]['memory_text']['text']
        mem_t_plus_1 = user_content[4]['memory_text']['text']
        print(f"\n[Sample {i}]")
        print(f"  mem_t (first 150 chars): {mem_t[:150]}...")
        print(f"  mem_t+1 (first 150 chars): {mem_t_plus_1[:150]}...")

Total 'Click on The Go button.' samples: 629
Unique (mem_t, mem_t+1) pairs: 1

=== Sample obs pairs for 'Click on The Go button.' ===

[Sample 0]
  mem_t (first 150 chars): <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <i...
  mem_t+1 (first 150 chars): <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <i...

[Sample 100]
  mem_t (first 150 chars): <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <i...
  mem_t+1 (first 150 chars): <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <i...

[Sample 300]
  mem_t (first 150 chars): <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header d

In [15]:
# 检查top 10高频action的obs唯一性
top_actions = click_action_dist.most_common(15)

print("Action repetition vs unique obs pairs:\n")
print(f"{'Action':<60} {'Count':>6} {'Unique Obs':>10} {'Ratio':>8}")
print("-" * 90)

for action, count in top_actions:
    action_samples = [s for s in cleaned_samples if s['messages'][2]['content'][0]['text'] == action]
    
    obs_pairs = []
    for s in action_samples:
        user_content = s['messages'][1]['content']
        mem_t = user_content[2]['memory_text']['text']
        mem_t_plus_1 = user_content[4]['memory_text']['text']
        obs_pairs.append((hash(mem_t), hash(mem_t_plus_1)))
    
    unique_pairs = len(set(obs_pairs))
    ratio = count / unique_pairs if unique_pairs > 0 else float('inf')
    
    print(f"{action[:58]:<60} {count:>6} {unique_pairs:>10} {ratio:>7.1f}x")

Action repetition vs unique obs pairs:

Action                                                        Count Unique Obs    Ratio
------------------------------------------------------------------------------------------
Click on The Go button.                                         629        626     1.0x
Click on the direction sign on the right side of the Go ic      187         44     4.2x
Click on the 'Forums' link at the top center.                   116          6    19.3x
Click on the 'Hot' button in the navigation menu.               110         97     1.1x
Click on the 'Alphabetical' navigation link.                    108          4    27.0x
Click on the 'Alphabetical' navigation link at the top cen       95          3    31.7x
Click on the 'My Orders' link.                                   82         59     1.4x
Click on the 'Forums' link.                                      64          4    16.0x
Click on the 'Next page' link.                                   59         4

In [16]:
# 按 (action, mem_t, mem_t+1) 去重
seen = set()
deduplicated_samples = []

for s in cleaned_samples:
    action = s['messages'][2]['content'][0]['text']
    user_content = s['messages'][1]['content']
    mem_t = user_content[2]['memory_text']['text']
    mem_t_plus_1 = user_content[4]['memory_text']['text']
    
    key = (action, hash(mem_t), hash(mem_t_plus_1))
    
    if key not in seen:
        seen.add(key)
        deduplicated_samples.append(s)

print(f"Before deduplication: {len(cleaned_samples)}")
print(f"After deduplication: {len(deduplicated_samples)}")
print(f"Removed: {len(cleaned_samples) - len(deduplicated_samples)} ({(len(cleaned_samples) - len(deduplicated_samples))/len(cleaned_samples)*100:.1f}%)")

# 新的分布
dedup_action_types = [s['action_type'] for s in deduplicated_samples]
dedup_dist = Counter(dedup_action_types)
print(f"\nAction type distribution after deduplication:")
for k, v in sorted(dedup_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v} ({v/len(deduplicated_samples)*100:.1f}%)")

Before deduplication: 7916
After deduplication: 6630
Removed: 1286 (16.2%)

Action type distribution after deduplication:
  Click: 3730 (56.3%)
  Type: 1321 (19.9%)
  Scroll Down: 798 (12.0%)
  Select Dropdown Option: 338 (5.1%)
  Search: 293 (4.4%)
  Hover: 51 (0.8%)
  unknown: 43 (0.6%)
  go_backward: 36 (0.5%)
  Scroll Up: 17 (0.3%)
  Press Enter: 2 (0.0%)
  Wait: 1 (0.0%)


In [17]:
# 检查所有action类型的重复情况
print("All action types - repetition analysis:\n")
print(f"{'Action Type':<25} {'Before':>8} {'After':>8} {'Removed':>8} {'Unique Actions':>15}")
print("-" * 70)

for action_type in sorted(dedup_dist.keys(), key=lambda x: -cleaned_dist[x]):
    before = cleaned_dist[action_type]
    after = dedup_dist[action_type]
    removed = before - after
    
    # 统计unique action文本数量
    type_samples = [s for s in deduplicated_samples if s['action_type'] == action_type]
    unique_actions = len(set(s['messages'][2]['content'][0]['text'] for s in type_samples))
    
    print(f"{action_type:<25} {before:>8} {after:>8} {removed:>7} ({removed/before*100:>4.1f}%) {unique_actions:>10}")

All action types - repetition analysis:

Action Type                 Before    After  Removed  Unique Actions
----------------------------------------------------------------------
Click                         4716     3730     986 (20.9%)       1315
Type                          1384     1321      63 ( 4.6%)       1285
Scroll Down                    896      798      98 (10.9%)          1
Select Dropdown Option         360      338      22 ( 6.1%)         90
Search                         303      293      10 ( 3.3%)        263
go_backward                    120       36      84 (70.0%)          1
Hover                           72       51      21 (29.2%)         37
unknown                         45       43       2 ( 4.4%)         41
Scroll Up                       17       17       0 ( 0.0%)          1
Press Enter                      2        2       0 ( 0.0%)          1
Wait                             1        1       0 ( 0.0%)          1


In [18]:
# 看看 Scroll Down, go_backward, Scroll Up 这些只有1个unique action的情况
# 它们的obs pair是否有差异？

for action_type in ['Scroll Down', 'go_backward', 'Scroll Up']:
    type_samples = [s for s in deduplicated_samples if s['action_type'] == action_type]
    
    # 检查obs pair的唯一性
    obs_pairs = []
    for s in type_samples:
        user_content = s['messages'][1]['content']
        mem_t = user_content[2]['memory_text']['text']
        mem_t_plus_1 = user_content[4]['memory_text']['text']
        obs_pairs.append((hash(mem_t), hash(mem_t_plus_1)))
    
    unique_obs = len(set(obs_pairs))
    
    print(f"{action_type}:")
    print(f"  Samples: {len(type_samples)}")
    print(f"  Unique obs pairs: {unique_obs}")
    print(f"  Action text: {type_samples[0]['messages'][2]['content'][0]['text'] if type_samples else 'N/A'}")
    print()

Scroll Down:
  Samples: 798
  Unique obs pairs: 798
  Action text: Scroll down the page.

go_backward:
  Samples: 36
  Unique obs pairs: 36
  Action text: Go back to the previous page.

Scroll Up:
  Samples: 17
  Unique obs pairs: 17
  Action text: Scroll up the page.



In [19]:
# 对每个action type，统计 (obs_t, action, obs_t+1) 三元组的去重情况
print("Deduplication analysis by (obs_t, action, obs_t+1) triplet:\n")
print(f"{'Action Type':<25} {'Raw':>8} {'Unique Triplets':>16} {'Removed':>12} {'Dedup Rate':>10}")
print("-" * 75)

total_raw = 0
total_dedup = 0

for action_type in sorted(cleaned_dist.keys(), key=lambda x: -cleaned_dist[x]):
    # 原始样本
    raw_samples = [s for s in cleaned_samples if s['action_type'] == action_type]
    raw_count = len(raw_samples)
    
    # 计算unique triplets
    triplets = set()
    for s in raw_samples:
        action = s['messages'][2]['content'][0]['text']
        user_content = s['messages'][1]['content']
        mem_t = user_content[2]['memory_text']['text']
        mem_t_plus_1 = user_content[4]['memory_text']['text']
        triplets.add((hash(mem_t), action, hash(mem_t_plus_1)))
    
    unique_count = len(triplets)
    removed = raw_count - unique_count
    dedup_rate = removed / raw_count * 100 if raw_count > 0 else 0
    
    total_raw += raw_count
    total_dedup += unique_count
    
    print(f"{action_type:<25} {raw_count:>8} {unique_count:>16} {removed:>8} ({dedup_rate:>5.1f}%)")

print("-" * 75)
print(f"{'TOTAL':<25} {total_raw:>8} {total_dedup:>16} {total_raw - total_dedup:>8} ({(total_raw - total_dedup)/total_raw*100:>5.1f}%)")

Deduplication analysis by (obs_t, action, obs_t+1) triplet:

Action Type                    Raw  Unique Triplets      Removed Dedup Rate
---------------------------------------------------------------------------
Click                         4716             3730      986 ( 20.9%)
Type                          1384             1321       63 (  4.6%)
Scroll Down                    896              798       98 ( 10.9%)
Select Dropdown Option         360              338       22 (  6.1%)
Search                         303              293       10 (  3.3%)
go_backward                    120               36       84 ( 70.0%)
Hover                           72               51       21 ( 29.2%)
unknown                         45               43        2 (  4.4%)
Scroll Up                       17               17        0 (  0.0%)
Press Enter                      2                2        0 (  0.0%)
Wait                             1                1        0 (  0.0%)
-----------------

In [20]:
# 分析Click类型中每个unique action的去重情况
click_raw_samples = [s for s in cleaned_samples if s['action_type'] == 'Click']

# 按action文本分组，统计raw count和unique triplet count
from collections import defaultdict

click_stats = defaultdict(lambda: {'raw': 0, 'triplets': set()})

for s in click_raw_samples:
    action = s['messages'][2]['content'][0]['text']
    user_content = s['messages'][1]['content']
    mem_t = user_content[2]['memory_text']['text']
    mem_t_plus_1 = user_content[4]['memory_text']['text']
    
    click_stats[action]['raw'] += 1
    click_stats[action]['triplets'].add((hash(mem_t), hash(mem_t_plus_1)))

# 转换为列表并排序
click_dedup_list = []
for action, stats in click_stats.items():
    raw = stats['raw']
    unique = len(stats['triplets'])
    removed = raw - unique
    click_dedup_list.append({
        'action': action,
        'raw': raw,
        'unique': unique,
        'removed': removed,
        'rate': removed / raw * 100 if raw > 0 else 0
    })

# 按removed数量排序
click_dedup_list.sort(key=lambda x: -x['removed'])

print("Click actions with most duplicates removed:\n")
print(f"{'Action':<65} {'Raw':>6} {'Unique':>6} {'Removed':>8}")
print("-" * 90)

for item in click_dedup_list[:30]:
    if item['removed'] > 0:
        print(f"{item['action'][:63]:<65} {item['raw']:>6} {item['unique']:>6} {item['removed']:>5} ({item['rate']:>4.1f}%)")

Click actions with most duplicates removed:

Action                                                               Raw Unique  Removed
------------------------------------------------------------------------------------------
Click on the direction sign on the right side of the Go icon.        187     44   143 (76.5%)
Click on the 'Forums' link at the top center.                        116      6   110 (94.8%)
Click on the 'Alphabetical' navigation link.                         108      4   104 (96.3%)
Click on the 'Alphabetical' navigation link at the top center.        95      3    92 (96.8%)
Click on the 'Forums' link.                                           64      4    60 (93.8%)
Click on the 'Filters' button.                                        46     21    25 (54.3%)
Click on the 'My Orders' link.                                        82     59    23 (28.0%)
Click on the 'MARKETING' section on the left sidebar.                 24      4    20 (83.3%)
Click on the 'REPORTS' 

In [21]:
# 检查相似action是否指向相同的obs
# 比如 'Forums' link 相关的多个action

forums_actions = [
    "Click on the 'Forums' link at the top center.",
    "Click on the 'Forums' link.",
    "Click on the 'Forums' link at the top of the page.",
    "Click on the 'Forums' link at the top left.",
]

alphabetical_actions = [
    "Click on the 'Alphabetical' navigation link.",
    "Click on the 'Alphabetical' navigation link at the top center.",
    "Click on the 'Alphabetical' navigation link at the top left.",
]

reports_actions = [
    "Click on the 'Reports' tab in the left sidebar menu.",
    "Click on the 'Reports' tab on the left sidebar.",
    "Click on the 'REPORTS' section on the left sidebar.",
]

# 收集所有obs pairs
def get_obs_pairs_for_actions(action_list):
    all_obs = set()
    for s in cleaned_samples:
        action = s['messages'][2]['content'][0]['text']
        if action in action_list:
            user_content = s['messages'][1]['content']
            mem_t = user_content[2]['memory_text']['text']
            mem_t_plus_1 = user_content[4]['memory_text']['text']
            all_obs.add((hash(mem_t), hash(mem_t_plus_1)))
    return all_obs

print("=== Checking if similar actions share same obs ===\n")

for name, action_list in [("Forums", forums_actions), ("Alphabetical", alphabetical_actions), ("Reports", reports_actions)]:
    print(f"--- {name} related actions ---")
    
    # 每个action的obs
    action_obs = {}
    for action in action_list:
        obs_set = set()
        count = 0
        for s in cleaned_samples:
            if s['messages'][2]['content'][0]['text'] == action:
                count += 1
                user_content = s['messages'][1]['content']
                mem_t = user_content[2]['memory_text']['text']
                mem_t_plus_1 = user_content[4]['memory_text']['text']
                obs_set.add((hash(mem_t), hash(mem_t_plus_1)))
        action_obs[action] = obs_set
        print(f"  {action[:50]}: {count} samples, {len(obs_set)} unique obs")
    
    # 检查obs重叠
    all_obs = set()
    for obs_set in action_obs.values():
        all_obs.update(obs_set)
    
    combined_unique = len(all_obs)
    sum_individual = sum(len(obs_set) for obs_set in action_obs.values())
    overlap = sum_individual - combined_unique
    
    print(f"  => Combined unique obs: {combined_unique}, Sum of individual: {sum_individual}, Overlap: {overlap}")
    print()

=== Checking if similar actions share same obs ===

--- Forums related actions ---
  Click on the 'Forums' link at the top center.: 116 samples, 6 unique obs
  Click on the 'Forums' link.: 64 samples, 4 unique obs
  Click on the 'Forums' link at the top of the page.: 16 samples, 4 unique obs
  Click on the 'Forums' link at the top left.: 13 samples, 3 unique obs
  => Combined unique obs: 6, Sum of individual: 17, Overlap: 11

--- Alphabetical related actions ---
  Click on the 'Alphabetical' navigation link.: 108 samples, 4 unique obs
  Click on the 'Alphabetical' navigation link at the: 95 samples, 3 unique obs
  Click on the 'Alphabetical' navigation link at the: 9 samples, 2 unique obs
  => Combined unique obs: 4, Sum of individual: 9, Overlap: 5

--- Reports related actions ---
  Click on the 'Reports' tab in the left sidebar men: 11 samples, 5 unique obs
  Click on the 'Reports' tab on the left sidebar.: 9 samples, 3 unique obs
  Click on the 'REPORTS' section on the left sidebar:

In [22]:
# 按 (obs_t, obs_t+1) 去重，保留第一个遇到的action描述
seen_obs = set()
dedup_by_obs_samples = []

for s in cleaned_samples:
    user_content = s['messages'][1]['content']
    mem_t = user_content[2]['memory_text']['text']
    mem_t_plus_1 = user_content[4]['memory_text']['text']
    
    obs_key = (hash(mem_t), hash(mem_t_plus_1))
    
    if obs_key not in seen_obs:
        seen_obs.add(obs_key)
        dedup_by_obs_samples.append(s)

print(f"Before (raw): {len(cleaned_samples)}")
print(f"After dedup by (action, obs_t, obs_t+1): {len(deduplicated_samples)}")
print(f"After dedup by (obs_t, obs_t+1) only: {len(dedup_by_obs_samples)}")
print(f"Additional removed: {len(deduplicated_samples) - len(dedup_by_obs_samples)}")

# 新的分布
obs_dedup_types = [s['action_type'] for s in dedup_by_obs_samples]
obs_dedup_dist = Counter(obs_dedup_types)
print(f"\nAction type distribution after obs-only deduplication:")
for k, v in sorted(obs_dedup_dist.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v} ({v/len(dedup_by_obs_samples)*100:.1f}%)")

Before (raw): 7916
After dedup by (action, obs_t, obs_t+1): 6630
After dedup by (obs_t, obs_t+1) only: 6298
Additional removed: 332

Action type distribution after obs-only deduplication:
  Click: 3477 (55.2%)
  Type: 1282 (20.4%)
  Scroll Down: 798 (12.7%)
  Select Dropdown Option: 331 (5.3%)
  Search: 293 (4.7%)
  Hover: 39 (0.6%)
  go_backward: 36 (0.6%)
  unknown: 22 (0.3%)
  Scroll Up: 17 (0.3%)
  Press Enter: 2 (0.0%)
  Wait: 1 (0.0%)


In [23]:
# 完整的去重对比表
print("Complete deduplication comparison:\n")
print(f"{'Action Type':<25} {'Raw':>7} {'Triplet Dedup':>14} {'Obs-Only Dedup':>15} {'Total Removed':>13}")
print("-" * 80)

total_raw = 0
total_triplet = 0
total_obs = 0

for action_type in sorted(cleaned_dist.keys(), key=lambda x: -cleaned_dist[x]):
    raw = cleaned_dist[action_type]
    triplet = dedup_dist.get(action_type, 0)
    obs = obs_dedup_dist.get(action_type, 0)
    
    total_raw += raw
    total_triplet += triplet
    total_obs += obs
    
    removed = raw - obs
    rate = removed / raw * 100 if raw > 0 else 0
    
    print(f"{action_type:<25} {raw:>7} {triplet:>14} {obs:>15} {removed:>8} ({rate:>5.1f}%)")

print("-" * 80)
total_removed = total_raw - total_obs
print(f"{'TOTAL':<25} {total_raw:>7} {total_triplet:>14} {total_obs:>15} {total_removed:>8} ({total_removed/total_raw*100:>5.1f}%)")

Complete deduplication comparison:

Action Type                   Raw  Triplet Dedup  Obs-Only Dedup Total Removed
--------------------------------------------------------------------------------
Click                        4716           3730            3477     1239 ( 26.3%)
Type                         1384           1321            1282      102 (  7.4%)
Scroll Down                   896            798             798       98 ( 10.9%)
Select Dropdown Option        360            338             331       29 (  8.1%)
Search                        303            293             293       10 (  3.3%)
go_backward                   120             36              36       84 ( 70.0%)
Hover                          72             51              39       33 ( 45.8%)
unknown                        45             43              22       23 ( 51.1%)
Scroll Up                      17             17              17        0 (  0.0%)
Press Enter                     2              2         

In [24]:
# 查看obs-only去重后Click的分布
click_dedup_samples = [s for s in dedup_by_obs_samples if s['action_type'] == 'Click']
click_dedup_actions = [s['messages'][2]['content'][0]['text'] for s in click_dedup_samples]
click_dedup_dist = Counter(click_dedup_actions)

print(f"Click samples after obs-only dedup: {len(click_dedup_samples)}")
print(f"Unique Click actions: {len(click_dedup_dist)}")

print(f"\nTop 30 most common Click actions:")
for action, count in click_dedup_dist.most_common(30):
    print(f"  {count:4d}x: {action[:75]}")

Click samples after obs-only dedup: 3477
Unique Click actions: 1216

Top 30 most common Click actions:
   625x: Click on The Go button.
    95x: Click on the 'Hot' button in the navigation menu.
    58x: Click on the 'My Orders' link.
    44x: Click on the 'Apply Filters' button.
    44x: Click on the direction sign on the right side of the Go icon.
    42x: Click on the 'Apply Filters' button next to the 'Cancel' button.
    42x: Click on the 'Next page' link.
    35x: Click on the 'All time' filter option.
    34x: Click on the 'Create submission' button.
    30x: Click on the 'Next Page' link.
    30x: Click on the 'Search' button.
    26x: Click on the 'Hot' button in the navigation menu at the top center.
    26x: Click on the 'Sign in' button.
    25x: Click on the 'Reset Filter' button.
    24x: Click on the 'All Reviews' option under 'User Content' section in the 'MARK
    23x: Click on the 'By Products' link under the 'Reviews' section.
    22x: Click on the 'Show Report' butt

In [25]:
# 确认 "Click on The Go button." 的625个样本obs都是唯一的
go_button_dedup = [s for s in dedup_by_obs_samples if s['messages'][2]['content'][0]['text'] == 'Click on The Go button.']

obs_pairs = set()
for s in go_button_dedup:
    user_content = s['messages'][1]['content']
    mem_t = user_content[2]['memory_text']['text']
    mem_t_plus_1 = user_content[4]['memory_text']['text']
    obs_pairs.add((hash(mem_t), hash(mem_t_plus_1)))

print(f"'Click on The Go button.' samples: {len(go_button_dedup)}")
print(f"Unique obs pairs: {len(obs_pairs)}")

# 看几个obs样本
print("\n=== Sample obs (first 200 chars) ===")
for i in [0, 200, 400, 600]:
    if i < len(go_button_dedup):
        s = go_button_dedup[i]
        user_content = s['messages'][1]['content']
        mem_t = user_content[2]['memory_text']['text']
        print(f"\n[{i}] mem_t: {mem_t[:200]}...")

'Click on The Go button.' samples: 625
Unique obs pairs: 625

=== Sample obs (first 200 chars) ===

[0] mem_t: <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <img id="0" data-bbox="10,13,30,30"> </img> </source...

[200] mem_t: <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <img id="0" data-bbox="10,13,30,30"> </img> </source...

[400] mem_t: <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <img id="0" data-bbox="10,13,30,30"> </img> </source...

[600] mem_t: <html data-bbox="0,0,1280,55"> <body data-bbox="0,0,1280,55"> <header data-bbox="0,0,1280,55"> <source type="image/svg+xml" data-bbox="10,17,0,22"> <img id="0" data-bbox="10,13,30,30"> </img> </source...


In [26]:
# 看完整的mem_t是否真的不同
go_button_dedup = [s for s in dedup_by_obs_samples if s['messages'][2]['content'][0]['text'] == 'Click on The Go button.']

# 用完整hash检查
full_mem_t_hashes = []
for s in go_button_dedup:
    user_content = s['messages'][1]['content']
    mem_t = user_content[2]['memory_text']['text']
    full_mem_t_hashes.append(hash(mem_t))

unique_mem_t = len(set(full_mem_t_hashes))
print(f"Total samples: {len(go_button_dedup)}")
print(f"Unique mem_t: {unique_mem_t}")

# 看mem_t的长度分布
mem_t_lengths = []
for s in go_button_dedup:
    user_content = s['messages'][1]['content']
    mem_t = user_content[2]['memory_text']['text']
    mem_t_lengths.append(len(mem_t))

print(f"\nmem_t length: min={min(mem_t_lengths)}, max={max(mem_t_lengths)}, mean={sum(mem_t_lengths)/len(mem_t_lengths):.0f}")

# 看不同位置的内容差异
print("\n=== Comparing different parts of mem_t ===")
sample_indices = [0, 1, 2, 100, 200]
for i in sample_indices:
    s = go_button_dedup[i]
    mem_t = s['messages'][1]['content'][2]['memory_text']['text']
    # 看中间部分
    mid = len(mem_t) // 2
    print(f"[{i}] len={len(mem_t)}, mid section: ...{mem_t[mid:mid+100]}...")

Total samples: 625
Unique mem_t: 625

mem_t length: min=3253, max=7267, mean=5326

=== Comparing different parts of mem_t ===
[0] len=3288, mid section: ...x="0,102,350,181"> <div data-bbox="16,118,318,47"> <h2 data-bbox="16,118,297,39"> Search Results </h...
[1] len=4804, mid section: ...eading, Berks County, 19610, United States </a> </li> <li data-bbox="0,369,350,80"> Filling Station ...
[2] len=4592, mid section: ...bbox="16,118,318,47"> <h2 data-bbox="16,118,297,39"> Search Results </h2> </div> <h4 data-bbox="16,1...
[100] len=3268, mid section: ...50,181"> <div data-bbox="16,118,318,47"> <h2 data-bbox="16,118,297,39"> Search Results </h2> <button...
[200] len=3388, mid section: ...69,20,20"> </img> </a> </div> </div> <div data-bbox="0,102,350,181"> <div data-bbox="16,118,318,47">...


In [27]:
# 查看 REPORTS 和 MARKETING 相关的action
reports_marketing_samples = [s for s in dedup_by_obs_samples 
                             if 'REPORTS' in s['messages'][2]['content'][0]['text'] 
                             or 'MARKETING' in s['messages'][2]['content'][0]['text']
                             or 'Reports' in s['messages'][2]['content'][0]['text']
                             or 'Marketing' in s['messages'][2]['content'][0]['text']]

print(f"REPORTS/MARKETING related samples: {len(reports_marketing_samples)}")

# 按action分组
action_counts = Counter(s['messages'][2]['content'][0]['text'] for s in reports_marketing_samples)
print("\nAction distribution:")
for action, count in action_counts.most_common():
    print(f"  {count:3d}x: {action}")

REPORTS/MARKETING related samples: 65

Action distribution:
   24x: Click on the 'All Reviews' option under 'User Content' section in the 'MARKETING'.
    8x: Click on the 'Bestsellers' report option under the 'Products' section in the 'REPORTS'.
    7x: Click on the 'REPORTS' section on the left sidebar.
    4x: Click on the 'MARKETING' section on the left sidebar.
    2x: Click on the 'Sales' report option under the 'Sales' section in the 'REPORTS'.
    2x: Click on the 'Orders' report option under the 'Sales' section in 'REPORTS'.
    2x: Click on the 'Reports' tab in the left sidebar menu.
    2x: Click on the 'Reports' link in the left sidebar under the 'CONTENT' section.
    2x: Click on the 'Content' link in the left sidebar under the 'MARKETING' section.
    1x: Click on the 'Reports' section on the left sidebar.
    1x: Click on the 'Bestsellers' report option under the 'Products' section in the 'Reports'.
    1x: Click on the 'Reports' link in the left sidebar menu.
    1x: C

In [28]:
# 检查这些action的obs是否有重叠
# 比如 "Click on the 'REPORTS' section" vs "Click on the 'Reports' tab" 是否指向相同的页面

reports_click_actions = [
    "Click on the 'REPORTS' section on the left sidebar.",
    "Click on the 'Reports' tab in the left sidebar menu.",
    "Click on the 'Reports' section on the left sidebar.",
    "Click on the 'Reports' link in the left sidebar menu.",
    "Click on the 'Reports' menu item on the left sidebar.",
    "Click on the 'Reports' tab in the left sidebar.",
]

marketing_click_actions = [
    "Click on the 'MARKETING' section on the left sidebar.",
    "Click on the 'Marketing' link in the left sidebar under 'Customers'.",
    "Click on the 'Marketing' tab in the left sidebar under 'Customers'.",
]

print("=== REPORTS section clicks ===")
for action in reports_click_actions:
    samples = [s for s in dedup_by_obs_samples if s['messages'][2]['content'][0]['text'] == action]
    if samples:
        print(f"\n{action}")
        print(f"  Count: {len(samples)}")
        # 看obs
        for i, s in enumerate(samples[:2]):
            mem_t = s['messages'][1]['content'][2]['memory_text']['text']
            print(f"  [{i}] mem_t (200 chars): {mem_t[:200]}...")

print("\n\n=== MARKETING section clicks ===")
for action in marketing_click_actions:
    samples = [s for s in dedup_by_obs_samples if s['messages'][2]['content'][0]['text'] == action]
    if samples:
        print(f"\n{action}")
        print(f"  Count: {len(samples)}")
        for i, s in enumerate(samples[:2]):
            mem_t = s['messages'][1]['content'][2]['memory_text']['text']
            print(f"  [{i}] mem_t (200 chars): {mem_t[:200]}...")

=== REPORTS section clicks ===

Click on the 'REPORTS' section on the left sidebar.
  Count: 7
  [0] mem_t (200 chars): <html data-bbox="0,0,1280,720"> <body data-bbox="0,0,1280,1618"> <div data-bbox="0,0,88,721"> <img id="0" title="Magento Admin Panel" data-bbox="27,17,35,41"> </img> <ul data-bbox="0,75,88,646"> <span...
  [1] mem_t (200 chars): <html data-bbox="0,0,1280,720"> <body data-bbox="0,0,1280,1618"> <div data-bbox="0,0,88,721"> <img id="0" title="Magento Admin Panel" data-bbox="27,17,35,41"> </img> <ul data-bbox="0,75,88,646"> <span...

Click on the 'Reports' tab in the left sidebar menu.
  Count: 2
  [0] mem_t (200 chars): <html data-bbox="0,0,1280,720"> <body data-bbox="0,0,1280,1618"> <div data-bbox="0,0,88,721"> <img id="0" title="Magento Admin Panel" data-bbox="27,17,35,41"> </img> <ul data-bbox="0,75,88,646"> <span...
  [1] mem_t (200 chars): <html data-bbox="0,0,1280,720"> <body data-bbox="0,0,1280,1618"> <div data-bbox="0,0,88,721"> <img id="0" title="Magento Admin P

In [29]:
# 检查这些"同一按钮不同描述"的obs是否真的不同
all_reports_marketing_actions = [
    "Click on the 'REPORTS' section on the left sidebar.",
    "Click on the 'Reports' tab in the left sidebar menu.",
    "Click on the 'Reports' section on the left sidebar.",
    "Click on the 'Reports' link in the left sidebar menu.",
    "Click on the 'Reports' menu item on the left sidebar.",
    "Click on the 'Reports' tab in the left sidebar.",
    "Click on the 'MARKETING' section on the left sidebar.",
    "Click on the 'Marketing' link in the left sidebar under 'Customers'.",
    "Click on the 'Marketing' tab in the left sidebar under 'Customers'.",
]

# 收集所有obs
all_obs_hashes = []
action_obs_map = []

for s in dedup_by_obs_samples:
    action = s['messages'][2]['content'][0]['text']
    if action in all_reports_marketing_actions:
        mem_t = s['messages'][1]['content'][2]['memory_text']['text']
        mem_t_plus_1 = s['messages'][1]['content'][4]['memory_text']['text']
        obs_hash = (hash(mem_t), hash(mem_t_plus_1))
        all_obs_hashes.append(obs_hash)
        action_obs_map.append((action, obs_hash))

print(f"Total samples for these actions: {len(all_obs_hashes)}")
print(f"Unique obs pairs: {len(set(all_obs_hashes))}")

# 检查是否有相同obs被不同action描述
from collections import defaultdict
obs_to_actions = defaultdict(set)
for action, obs_hash in action_obs_map:
    obs_to_actions[obs_hash].add(action)

# 找出同一个obs有多个action描述的情况
multi_action_obs = {obs: actions for obs, actions in obs_to_actions.items() if len(actions) > 1}
print(f"\nObs with multiple action descriptions: {len(multi_action_obs)}")

if multi_action_obs:
    print("\nExamples:")
    for obs, actions in list(multi_action_obs.items())[:5]:
        print(f"  Obs hash: {obs}")
        for a in actions:
            print(f"    - {a}")

Total samples for these actions: 19
Unique obs pairs: 19

Obs with multiple action descriptions: 0
